<a href="https://colab.research.google.com/github/giuseppe-maffucci-01/happo-training-mlp/blob/main/HAPPO_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
!pip install gymnasium
!pip install git+https://github.com/giorgiofranceschelli/Gymnasium-Stag-Hunt.git

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions.categorical import Categorical
import math
import json
import gymnasium as gym
import gymnasium_stag_hunt
from gymnasium_stag_hunt.envs.gym.hunt import HuntEnv
import numpy as np
from gymnasium_stag_hunt.src.utils import place_entity_in_unoccupied_cell, spawn_plants
import functools
from torch.nn.utils import clip_grad_norm_
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import LambdaLR

  Cloning https://github.com/giorgiofranceschelli/Gymnasium-Stag-Hunt.git to /tmp/pip-req-build-xkyx5ml4
  Running command git clone --filter=blob:none --quiet https://github.com/giorgiofranceschelli/Gymnasium-Stag-Hunt.git /tmp/pip-req-build-xkyx5ml4
  Resolved https://github.com/giorgiofranceschelli/Gymnasium-Stag-Hunt.git to commit 39b6b5d6c653dd38655288930c660faff9b2fb01
  Preparing metadata (setup.py) ... done


**Testing Configuration Parameter**: since this test notebook is written for the training configuration done in the training notebook, please if you want to try this notebook modify just the parameters below

In [28]:
# -----------------------------------------------------------------------------
# Test Configuration Parameter
# -----------------------------------------------------------------------------
MAX_ITERATIONS: int = 40 #the max lenght of a game, therefore the game can end before or after this
INTERVAL: int = 800 #ms between 1 frame and another in the animations

**Actor, critic, wrappers and adapter**: they are the same (unless the adapter which now takes into accont just the fact that we may have or not the streak shown to the agents) as the training notebook

In [29]:
from typing import Tuple, Union


class Actor(nn.Module):
    """
    Local Actor Network for individual policy parameterization.
    """

    def __init__(self, input_dim: int, output_dim: int, lr: float = 3e-4):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.hidden_layers = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
        )
        self.out = nn.Linear(128, output_dim)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=lr) #optimizer, used later

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.norm(x)
        x = self.hidden_layers(x)
        return self.out(x)  # #out[0] = UP, out[1] = DOWN, out[2] = LEFT, out[3] = RIGHT, out[4] = STAY, see torch.multinomial in self.generate_action

    def generate_action(self, input_obs: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
        logits = self(input_obs)
        probs = F.softmax(logits / temperature, dim=-1)
        action = torch.multinomial(probs, num_samples=1)
        return action #shape: (num_envs, 1)

    def evaluate_actions(self, obs: torch.Tensor, actions: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        logits = self(obs)
        dist = Categorical(logits=logits)

        if isinstance(actions, torch.Tensor):
            actions = actions.squeeze(-1).long()
        else:
            actions = torch.tensor(actions, device=logits.device).long()

        log_probs = dist.log_prob(actions).unsqueeze(-1) #shape: (B, 1)
        entropy = dist.entropy().unsqueeze(-1) #shape: (B, 1)

        return log_probs, entropy

    def compute_actor_loss(
            self,
            new_log_prob: torch.Tensor,
            old_log_prob_batch: torch.Tensor,
            factor_m_batch: torch.Tensor,
            adv_batch: torch.Tensor,
            entropy: torch.Tensor,
            eps_clip: float,
            entropy_coef: float,
        ) -> torch.Tensor:
        """
        Computes HAPPO clipped surrogate policy loss with entropy regularization.
        """

        imp_weights = torch.exp(new_log_prob - old_log_prob_batch) #importance sampling ratio for the current agent

        effective_adv = factor_m_batch * adv_batch #advantage weighted by factor M comuled from previous agents

        # PPO-Clip objective
        surr1 = imp_weights * effective_adv
        surr2 = torch.clamp(imp_weights, 1.0 - eps_clip, 1.0 + eps_clip) * effective_adv

        policy_loss = -torch.min(surr1, surr2).mean() # - E(L_clip)
        entropy_loss = -entropy_coef * entropy.mean() # - coeff*E(H)

        return policy_loss + entropy_loss # L_actor = -E(L_clip + coeff*H) = - (E(L_clip) + coeff*E(H))



class Critic(nn.Module):
    """
    Centralized Critic Network evaluating global system state values V(s).
    """

    def __init__(self, input_dim: int, lr: float = 1e-3):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.hidden_layers = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
        )
        self.out = nn.Linear(128, 1)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=lr)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.norm(x)
        x = self.hidden_layers(x)
        return self.out(x)

    def huber_loss(self, e: torch.Tensor, delta: float = 10.0) -> torch.Tensor:
        """
        Calculates element-wise Huber loss function.
        """
        abs_e = torch.abs(e)
        return torch.where(abs_e <= delta, 0.5 * (e ** 2), delta * (abs_e - 0.5 * delta))

    def compute_critic_loss(
        self,
        old_v_preds_batch: torch.Tensor,
        v_preds: torch.Tensor,
        returns_batch: torch.Tensor,
        eps_clip: float,
        huber_delta: float,
        use_clipped_value_loss: bool,
    ) -> torch.Tensor:
        """
        Calculates centralized critic loss with optional value clipping.
        """
        if use_clipped_value_loss:
            v_pred_clipped = old_v_preds_batch + torch.clamp(v_preds - old_v_preds_batch, -eps_clip, eps_clip)
            error_original = returns_batch - v_preds
            error_clipped = returns_batch - v_pred_clipped

            loss_original = self.huber_loss(error_original, huber_delta)
            loss_clipped = self.huber_loss(error_clipped, huber_delta)
            critic_loss = torch.max(loss_original, loss_clipped).mean()
        else:
            error = returns_batch - v_preds
            critic_loss = self.huber_loss(error, huber_delta).mean()
        return critic_loss



class StagHuntWrapper(gym.Wrapper): #this wrapper is just for a single gym environment
    def __init__(self, env, min_stag_dist=2):
        super().__init__(env)
        self.min_stag_dist = min_stag_dist
        self.action_space = gym.spaces.MultiDiscrete([5, 5])

    def reset(self, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        game = self.env.unwrapped.game
        grid_dims = game.GRID_DIMENSIONS

        game.A_AGENT = place_entity_in_unoccupied_cell(grid_dims=grid_dims, used_coordinates=[])
        game.B_AGENT = place_entity_in_unoccupied_cell(grid_dims=grid_dims, used_coordinates=[game.A_AGENT])

        ax, ay = int(game.A_AGENT[0]), int(game.A_AGENT[1])
        bx, by = int(game.B_AGENT[0]), int(game.B_AGENT[1])

        while True: #place the stag at least 2 units far (in terms of manhattan distance) from the agents
            stag_pos = place_entity_in_unoccupied_cell(
                grid_dims=grid_dims,
                used_coordinates=[game.A_AGENT, game.B_AGENT]
            )
            sx, sy = int(stag_pos[0]), int(stag_pos[1])
            dist_a = abs(sx - ax) + abs(sy - ay)
            dist_b = abs(sx - bx) + abs(sy - by)
            if dist_a >= self.min_stag_dist and dist_b >= self.min_stag_dist:
                break

        game.STAG = stag_pos
        game.PLANTS = spawn_plants(
            grid_dims=grid_dims,
            how_many=game._forage_quantity,
            used_coordinates=game.AGENTS + [game.STAG],
        )
        game._tagged_plants = []
        game._timestep = 0

        obs_list = game.get_observation() #local observations of the single environment

        return obs_list, info #to be coherent with gymnasium, it wants obs and info in the reset() function

    def step(self, actions_sent): #for each environment actions_sent is [act_a, act_b]
        next_obs_list, rewards, terminated, truncated, info = self.env.step(actions_sent)
        game = self.env.unwrapped.game

        r_a, r_b = rewards
        stag_caught = (r_a == game._stag_reward and r_b == game._stag_reward)
        agent_mauled = (r_a == game._mauling_punishment or r_b == game._mauling_punishment)

        custom_done = (stag_caught or agent_mauled)

        if custom_done: #reset to start a new game
            next_obs_list, info = self.reset()

        info['rewards_a_b'] = np.array([r_a, r_b], dtype=np.float32)
        info['custom_done'] = custom_done
        global_reward = float(r_a + r_b)

        return next_obs_list, global_reward, terminated, truncated, info #gymnasium wants: obs, reward, terminated, truncated, info


class HarvestWrapper(gym.Wrapper): #this wrapper is just for a single gym environment
    def __init__(self, env, min_plant_dist=2):
        super().__init__(env)
        self.min_plant_dist = min_plant_dist
        self.action_space = gym.spaces.MultiDiscrete([5, 5])

    def reset(self, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        game = self.env.unwrapped.game
        grid_dims = game.GRID_DIMENSIONS

        game.A_AGENT = np.array(place_entity_in_unoccupied_cell(grid_dims=grid_dims, used_coordinates=[]), dtype=np.uint8)
        game.B_AGENT = np.array(place_entity_in_unoccupied_cell(grid_dims=grid_dims, used_coordinates=[game.A_AGENT]), dtype=np.uint8)

        ax, ay = int(game.A_AGENT[0]), int(game.A_AGENT[1])
        bx, by = int(game.B_AGENT[0]), int(game.B_AGENT[1])

        for plant_number in range(game._max_plants):
            while True:
                plant_pos = place_entity_in_unoccupied_cell(
                    grid_dims=grid_dims,
                    used_coordinates=[game.A_AGENT, game.B_AGENT]
                )
                sx, sy = int(plant_pos[0]), int(plant_pos[1])
                dist_a = abs(sx - ax) + abs(sy - ay)
                dist_b = abs(sx - bx) + abs(sy - by)
                if dist_a >= self.min_plant_dist and dist_b >= self.min_plant_dist:
                    break

            game._plants[plant_number] = np.array(plant_pos, dtype=np.uint8)

        game._maturity_flags = [False] * game._max_plants
        game._timestep = 0

        obs_list = [np.array(o, dtype=np.uint8) for o in game.get_observation()]

        return obs_list, info

    def step(self, actions_sent):
        next_obs_list, rewards, terminated, truncated, info = self.env.step(actions_sent)
        next_obs_list = [np.array(o, dtype=np.uint8) for o in next_obs_list]

        r_a, r_b = rewards
        global_reward = float(r_a + r_b)

        info['rewards_a_b'] = np.array([r_a, r_b], dtype=np.float32)
        info['custom_done'] = terminated or truncated

        return next_obs_list, global_reward, terminated, truncated, info




class EscalationWrapper(gym.Wrapper):  #this wrapper is just for a single gym environment
    def __init__(self, env, min_mark_dist=2, gamma=0.99, alpha=0.1, max_streak=10):
        super().__init__(env)
        self.min_mark_dist = min_mark_dist
        self.gamma = gamma
        self.alpha = alpha
        self.max_streak = max_streak
        self.action_space = gym.spaces.MultiDiscrete([5, 5])
        self.current_phi = 0.0


    def _calc_phi(self, game): #compute the manhattan distances from agents and mark and sum them
        ax, ay = int(game.A_AGENT[0]), int(game.A_AGENT[1])
        bx, by = int(game.B_AGENT[0]), int(game.B_AGENT[1])
        mx, my = int(game._mark[0]), int(game._mark[1])

        dist_a = abs(ax - mx) + abs(ay - my)
        dist_b = abs(bx - mx) + abs(by - my)
        return float(dist_a + dist_b)


    def reset(self, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        game = self.env.unwrapped.game
        grid_dims = game.GRID_DIMENSIONS

        game.A_AGENT = np.array(place_entity_in_unoccupied_cell(grid_dims=grid_dims, used_coordinates=[]), dtype=np.uint8)
        game.B_AGENT = np.array(place_entity_in_unoccupied_cell(grid_dims=grid_dims, used_coordinates=[game.A_AGENT]), dtype=np.uint8)

        ax, ay = int(game.A_AGENT[0]), int(game.A_AGENT[1])
        bx, by = int(game.B_AGENT[0]), int(game.B_AGENT[1])

        while True:
            mark_pos = place_entity_in_unoccupied_cell(
                grid_dims=grid_dims,
                used_coordinates=[game.A_AGENT, game.B_AGENT]
            )
            mx, my = int(mark_pos[0]), int(mark_pos[1])
            dist_a = abs(mx - ax) + abs(my - ay)
            dist_b = abs(mx - bx) + abs(my - by)
            if dist_a >= self.min_mark_dist and dist_b >= self.min_mark_dist:
                break

        game._mark = np.array(mark_pos, dtype=np.uint8) #assigns the mark
        game._streak = 0 #reset the streak
        game._streak_active = False
        game._timestep = 0

        self.current_phi = self._calc_phi(game) #initial potential (Phi(o_0))

        obs_list = game.get_observation()
        info = {
          'streak': game._streak,
          'custom_done': False
        }
        return obs_list, info

    def step(self, actions_sent):
        next_obs_list, rewards, terminated, truncated, info = self.env.step(actions_sent)
        game = self.env.unwrapped.game
        # print(f"env rewards: {rewards}")
        r_a, r_b = rewards
        base_reward = float(r_a + r_b)

        next_phi = self._calc_phi(game)
        f_shaping = self.current_phi - (self.gamma * next_phi) #potential-based Reward Shaping
        self.current_phi = next_phi

        global_reward = base_reward + (self.alpha * f_shaping)
        # print(f"game_streak_active: {game._streak_active}")
        streak_broken = (not game._streak_active and (r_a < 0 or r_b < 0)) #check custom end game conditions
        streak_completed = (game._streak >= self.max_streak)

        custom_done = streak_broken or streak_completed or terminated or truncated

        if custom_done: #game ended: reset environment
            next_obs_list, info = self.reset()
        else:
            next_obs_list = game.get_observation() #coherent with gymnasium

        info["rewards_a_b"] = np.array([r_a, r_b], dtype=np.float32)
        info["custom_done"] = custom_done
        info["streak"] = game._streak

        return next_obs_list, global_reward, terminated, truncated, info


def happo_test_vector_adapter(env, show_streak_to_actors=True): #needed just to manage the show_streak_to_actors for the escalation mode
    orig_reset = env.reset
    orig_step = env.step

    @functools.wraps(orig_reset)
    def custom_reset(*args, **kwargs):
        obs, infos = orig_reset(*args, **kwargs)
        is_escalation = getattr(env.unwrapped, "game_title", "").lower() == "escalation" #if escalation we have have additional things to consider
        obs_list = obs.tolist() if isinstance(obs, np.ndarray) else obs
        if is_escalation and show_streak_to_actors:
            infos['streak']
            obs_list[0].append(infos['streak'])
            obs_list[1].append(infos['streak'])
        return obs_list, infos

    @functools.wraps(orig_step)
    def custom_step(actions, *args, **kwargs):
        obs, rewards, terminations, truncations, infos = orig_step(actions, *args, **kwargs)
        is_escalation = getattr(env.unwrapped, "game_title", "").lower() == "escalation"
        obs_list = obs.tolist() if isinstance(obs, np.ndarray) else obs
        if is_escalation and show_streak_to_actors:
            infos['streak']

            obs_list[0].append(infos['streak'])
            obs_list[1].append(infos['streak'])
        return obs_list, rewards, terminations, truncations, infos

    env.reset = custom_reset
    env.step = custom_step
    return env

def make_env(game_name, game_configs):
    env = gym.make(f"StagHunt-{game_name}-v0", **game_configs)
    if game_name == "Hunt":
        return StagHuntWrapper(env, min_stag_dist=2)
    elif game_name == "Harvest":
        return HarvestWrapper(env, min_plant_dist=2)
    elif game_name == "Escalation":
        return EscalationWrapper(env, min_mark_dist=2)
    else:
        raise ValueError(f"Gioco non riconosciuto: {game_name}")

**Function to load the agents trained in the training notebook**: to understand better the function, take into account that the dictionary named **github_checkpoint** downloaded has this structure:

github_checkpoint = {
    
    'actors_state_dict': [agent.state_dict() for agent in agents], <---actors weights
  
    'actors_optimizer_state_dict': [agent.optimizer.state_dict() for agent in agents],  <--- actors state dicts

    'critic_state_dict': critic.state_dict(), <--- critic weights (we have just 1 critic)
    'critic_optimizer_state_dict': critic.optimizer.state_dict(), <--- critic state dict

    'config': {  <---useful configurations parameters to recreate the agents
        'num_agents': NUM_AGENTS,
        'input_dim': input_dim,
        'output_dim': OUTPUT_DIM,
        'centr_input_dim': centr_input_dim,
    }
}

In [36]:
import os
import torch
from typing import List

def load_actors(github_checkpoint_path: str, github_weights_url: str) -> List[Actor]:
    if not os.path.exists(github_checkpoint_path):
        os.system(f'wget -O "{github_checkpoint_path}" "{github_weights_url}"')

    github_checkpoint = torch.load(github_checkpoint_path, map_location="cpu")

    input_dim = github_checkpoint["config"]["input_dim"]
    output_dim = github_checkpoint["config"]["output_dim"]
    num_agents = github_checkpoint["config"]["num_agents"]

    agents = [Actor(input_dim, output_dim).to("cpu") for _ in range(num_agents)] #agents initialization, for the time being I will run all on CPU
    for idx, agent in enumerate(agents):
        agent.load_state_dict(github_checkpoint["actors_state_dict"][idx])
        agent.eval()  # this is needed when I'll try to extend this project for LLMs

    return agents

In [31]:
def generate_markov_process(env, agents, max_iterations, game_type):
  history_obs = []
  obs, info = env.reset()

  max_iterations = 40
  history_obs.append(obs[0])

  # Mapping: actions -> displacements (dx, dy)
  # 0: LEFT, 1: DOWN, 2: RIGHT, 3: UP, 4: STAY
  ACTION_TO_DELTA = {
      0: [-1, 0],
      1: [0, 1],
      2: [1, 0],
      3: [0, -1],
      4: [0, 0]
  }

  for iteration in range(max_iterations):

      # generate actions
      actions = []
      with torch.no_grad():
          for idx, agent in enumerate(agents):
              agent_obs = torch.tensor(obs[idx], dtype=torch.float32).unsqueeze(0)
              action_tensor = agent.generate_action(agent_obs)
              actions.append(action_tensor.item())

      # environment step
      next_obs, rewards, terminated, truncated, next_info = env.step(actions)

      # save a copy of the current obs in the hostory list
      current_obs = obs[0].copy()
      history_obs.append(next_obs[0].copy())

      # custom done at the end of a game
      if next_info.get("custom_done", False):

          # get displacement vectors
          delta_A = ACTION_TO_DELTA.get(actions[0], [0, 0])
          delta_B = ACTION_TO_DELTA.get(actions[1], [0, 0])

          # position must be between 0 and 4
          new_A_pos = [min(max(int(x) + dx, 0), 4) for x, dx in zip(obs[0][0:2], delta_A)]
          new_B_pos = [min(max(int(x) + dx, 0), 4) for x, dx in zip(obs[0][2:4], delta_B)]

          # update A and B position
          current_obs[0:2] = new_A_pos
          current_obs[2:4] = new_B_pos

          # manage the stag based on game type and reward
          single_rewards = next_info.get("rewards_a_b", [0, 0])

          if game_type == "Hunt":
              if rewards == 10:
                  # Entrambi catturano lo Stag: lo Stag va nella posizione dell'Agente A
                  new_stag_pos = new_A_pos.copy()
                  current_obs[4:6] = new_stag_pos


              elif rewards == -5:
                  # Uno solo tenta la cattura fallendo: lo Stag si sposta su chi ci ha provato
                  if single_rewards[0] == -5:
                      new_stag_pos = new_A_pos.copy()
                  elif single_rewards[1] == -5:
                      new_stag_pos = new_B_pos.copy()
                  else:
                      new_stag_pos = current_obs[4:6]

                  current_obs[4:6] = new_stag_pos


              # override the last saved obs in the history with the correct one
              history_obs[-1] = current_obs

              break

          elif game_type == "Harvest":
              # override the last saved obs in the history with the correct one
              history_obs[-1] = current_obs

              break

          elif game_type == "Escalation": #here, in the last frame, you can see whether the agents split up, but you can't see which cell the stag moves into because the environment resets as soon as I finish
              # override the last saved obs in the history with the correct one
              history_obs[-1] = current_obs

              break

      # update obs
      obs = next_obs
      info = next_info

  return history_obs

**Functions to create the animations**

In [32]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
from collections import defaultdict

def animate_grid_observations_hunt(history_obs, grid_size=5, interval=400):
    """HTML animation for the Hunt mode.

    Parameters:
    - history_obs: obs list in format: [xA, yA, xB, yB, xS, yS, xP1, yP1, xP2, yP2]
    - grid_size: (default 5x5)
    - interval: time between one frame and the next
    """
    fig, ax = plt.subplots(figsize=(5, 5))

    # mapping entities
    entity_info = [
        ('A', 'blue'),       # Agent A: components [0, 1]
        ('B', 'green'),      # Agent B: components [2, 3]
        ('S', 'darkred'),    # Stag: components [4, 5]
        ('P1', 'darkgreen'), # Plant 1: components [6, 7]
        ('P2', 'darkgreen')  # Plant 2: components [8, 9]
    ]

    def update(frame):
        ax.clear()
        obs = history_obs[frame]

        # grid settings
        ax.set_xlim(0, grid_size)
        ax.set_ylim(0, grid_size)
        ax.set_xticks(range(grid_size + 1))
        ax.set_yticks(range(grid_size + 1))
        ax.grid(True, which='both', color='gray', linestyle='-', linewidth=1.5)
        ax.set_title(f"Step / Frame: {frame + 1} di {len(history_obs)}", fontsize=14, fontweight='bold')

        # the origin (0,0) is in the top left corner
        ax.invert_yaxis()

        # entities present on the same cell
        cell_occupants = defaultdict(list)

        for idx, (label, color) in enumerate(entity_info):
            x = obs[2 * idx]
            y = obs[2 * idx + 1]
            cell_occupants[(x, y)].append(label)

        # text in the cell
        for (x, y), labels in cell_occupants.items():
            if 0 <= x < grid_size and 0 <= y < grid_size:
                text_content = "\n".join(labels) if len(labels) > 2 else ", ".join(labels)
                ax.text(
                    x + 0.5, y + 0.5,
                    text_content,
                    ha='center', va='center',
                    fontsize=12, fontweight='bold',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="orange", alpha=0.8)
                )


    anim = animation.FuncAnimation(fig, update, frames=len(history_obs), interval=interval)
    plt.close(fig)

    display(HTML(anim.to_jshtml()))



def animate_grid_observations_harvest(history_obs, grid_size=5, interval=400):
    """HTML animation for the Harvest mode.

    Parameters:
    - history_obs: obs list in format: [xA, yA, xB, yB, xP1, yP1, ripeP1, xP2, yP2, ripeP2, xP3, yP3, ripeP3, xP4, yP4, ripeP4]
    - grid_size: (default 5x5)
    - interval: time between one frame and the next
    """
    fig, ax = plt.subplots(figsize=(5, 5))

    def update(frame):
        ax.clear()
        obs = history_obs[frame]

        # grid settings
        ax.set_xlim(0, grid_size)
        ax.set_ylim(0, grid_size)
        ax.set_xticks(range(grid_size + 1))
        ax.set_yticks(range(grid_size + 1))
        ax.grid(True, which="both", color="gray", linestyle="-", linewidth=1.5)
        ax.set_title(
            f"Step / Frame: {frame + 1} di {len(history_obs)}",
            fontsize=14,
            fontweight="bold",
        )

        ax.invert_yaxis()  # origin (0,0) in the left side corner

        cell_occupants = defaultdict(list)

        # agents position (A: obs[0,1], B: obs[2,3])
        cell_occupants[(obs[0], obs[1])].append("A")
        cell_occupants[(obs[2], obs[3])].append("B")

        # plants (we consider 4 planets in the test)
        # triplet: [x, y, ripe] where ripe non-zero means the plant is ripe
        for i in range(4):
            idx = 4 + (i * 3)
            px, py, ripe = obs[idx], obs[idx + 1], obs[idx + 2]

            # show if a plant is ripe or not
            status_symbol = "★" if ripe else "·"
            plant_label = f"P{i+1}({status_symbol})"

            cell_occupants[(px, py)].append(plant_label)

        # text in cells
        for (x, y), labels in cell_occupants.items():
            if 0 <= x < grid_size and 0 <= y < grid_size:
                text_content = (
                    "\n".join(labels) if len(labels) > 2 else ", ".join(labels)
                )

                ax.text(
                    x + 0.5,
                    y + 0.5,
                    text_content,
                    ha="center",
                    va="center",
                    fontsize=10,
                    fontweight="bold",
                    bbox=dict(
                        boxstyle="round,pad=0.3",
                        facecolor="lightyellow",
                        edgecolor="orange",
                        alpha=0.8,
                    ),
                )

    anim = animation.FuncAnimation(
        fig, update, frames=len(history_obs), interval=interval
    )
    plt.close(fig)

    display(HTML(anim.to_jshtml()))



def animate_grid_observations_escalation(
    history_obs, grid_size=5, interval=400
):
    """HTML animation for the Escalation mode.

    Parameters:
    - history_obs: list of obs on the format [xA, yA, xB, yB, xS, yS, (opzionale: extra_info)]
    - grid_size: grid dimentions (default 5x5)
    - interval: times between a frame and the next
    """
    fig, ax = plt.subplots(figsize=(5, 5))

    # entities in the first 6 indices (0,1 -> A | 2,3 -> B | 4,5 -> S)
    entity_info = [("A", "blue"), ("B", "green"), ("S", "darkred")]

    def update(frame):
        ax.clear()
        obs = history_obs[frame]

        # set the grid
        ax.set_xlim(0, grid_size)
        ax.set_ylim(0, grid_size)
        ax.set_xticks(range(grid_size + 1))
        ax.set_yticks(range(grid_size + 1))
        ax.grid(True, which="both", color="gray", linestyle="-", linewidth=1.5)
        ax.set_title(
            f"Step / Frame: {frame + 1} di {len(history_obs)}",
            fontsize=14,
            fontweight="bold",
        )

        ax.invert_yaxis()  # keeps (0,0) in the up left corner

        cell_occupants = defaultdict(list)

        # read just the first 6 numbers (we are in escalation and the last number (if present) is just the lenght of the streak)
        for idx, (label, color) in enumerate(entity_info):
            x = obs[2 * idx]
            y = obs[2 * idx + 1]
            cell_occupants[(x, y)].append(label)

        # text in cells
        for (x, y), labels in cell_occupants.items():
            if 0 <= x < grid_size and 0 <= y < grid_size:
                text_content = (
                    "\n".join(labels) if len(labels) > 2 else ", ".join(labels)
                )

                ax.text(
                    x + 0.5,
                    y + 0.5,
                    text_content,
                    ha="center",
                    va="center",
                    fontsize=12,
                    fontweight="bold",
                    bbox=dict(
                        boxstyle="round,pad=0.3",
                        facecolor="lightyellow",
                        edgecolor="orange",
                        alpha=0.8,
                    ),
                )

    anim = animation.FuncAnimation(
        fig, update, frames=len(history_obs), interval=interval
    )
    plt.close(fig)

    display(HTML(anim.to_jshtml()))

**Hunt / Harvest / Escalation testing**

In [33]:
from dataclasses import dataclass, field
from typing import Dict, Any

ENV_CONFIGS: Dict[str, Any] = {
    "obs_type": "coords",  # Coordinates format: [A_x, A_y, B_x, B_y, S_x, S_y, P1_x, P1_y, ...]
    "enable_multiagent": True,
    "load_renderer": False,
}

print("="*50)
print("Testing on Hunt environment")
print("="*50)
GITHUB_WEIGHTS_URL = "https://github.com/giuseppe-maffucci-01/happo-training-mlp/releases/download/v1.0.0/happo_multi_agent_checkpoint_hunt.pth"
GITHUB_CHECKPOINT_PATH = "happo_multi_agent_checkpoint_hunt.pth"
env = make_env("Hunt", ENV_CONFIGS)
env = happo_test_vector_adapter(env, show_streak_to_actors=True)
agents = load_actors(GITHUB_CHECKPOINT_PATH, GITHUB_WEIGHTS_URL)
hunt_history_obs = generate_markov_process(env, agents, MAX_ITERATIONS, "Hunt")
animate_grid_observations_hunt(hunt_history_obs, grid_size=5, interval=INTERVAL)

print("="*50)
print("Testing on Harvest environment")
print("="*50)
GITHUB_WEIGHTS_URL = "https://github.com/giuseppe-maffucci-01/happo-training-mlp/releases/download/v1.0.0/happo_multi_agent_checkpoint_harvest.pth"
GITHUB_CHECKPOINT_PATH = "happo_multi_agent_checkpoint_harvest.pth"
env = make_env("Harvest", ENV_CONFIGS)
env = happo_test_vector_adapter(env, show_streak_to_actors=True)
agents = load_actors(GITHUB_CHECKPOINT_PATH, GITHUB_WEIGHTS_URL)
harvest_history_obs = generate_markov_process(env, agents, MAX_ITERATIONS, "Harvest")
animate_grid_observations_harvest(harvest_history_obs, grid_size=5, interval=INTERVAL)

print("="*50)
print("Testing on Escalation environment")
print("="*50)
GITHUB_WEIGHTS_URL = "https://github.com/giuseppe-maffucci-01/happo-training-mlp/releases/download/v1.0.0/happo_multi_agent_checkpoint_escalation.pth"
GITHUB_CHECKPOINT_PATH = "happo_multi_agent_checkpoint_escalation.pth"
env = make_env("Escalation", ENV_CONFIGS)
env = happo_test_vector_adapter(env, show_streak_to_actors=True)
agents = load_actors(GITHUB_CHECKPOINT_PATH, GITHUB_WEIGHTS_URL)
escalation_history_obs = generate_markov_process(env, agents, MAX_ITERATIONS, "Escalation")
animate_grid_observations_escalation(escalation_history_obs, grid_size=5, interval=INTERVAL)

Testing on Hunt environment


/usr/local/lib/python3.13/dist-packages/gymnasium/utils/passive_env_checker.py:244: UserWarning: WARN: The reward returned by `step()` must be a float, int, np.integer or np.floating, actual type: <class 'tuple'>
  logger.warn(


Testing on Harvest environment


/usr/local/lib/python3.13/dist-packages/gymnasium/utils/passive_env_checker.py:133: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be uint8, actual type: int64
  logger.warn(
/usr/local/lib/python3.13/dist-packages/gymnasium/utils/passive_env_checker.py:157: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/usr/local/lib/python3.13/dist-packages/gymnasium/utils/passive_env_checker.py:133: UserWarning: WARN: The obs returned by the `step()` method was expecting numpy array dtype to be uint8, actual type: int64
  logger.warn(
/usr/local/lib/python3.13/dist-packages/gymnasium/utils/passive_env_checker.py:157: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/usr/local/lib/python3.13/dist-packages/gymnasium/utils/passive_env_chec

Testing on Escalation environment


/usr/local/lib/python3.13/dist-packages/gymnasium/utils/passive_env_checker.py:244: UserWarning: WARN: The reward returned by `step()` must be a float, int, np.integer or np.floating, actual type: <class 'tuple'>
  logger.warn(
